# Trilha rápida do analista de dados

**Python, SQL e DAX — Dr. Osvaldo L. Santos-Pereira**

Este notebook acompanha a [videoaula](https://youtu.be/dLDmXewxwpA) e apresenta operações recorrentes de manipulação tabular. Os dados são sintéticos; a semente e a data de referência são fixadas para garantir reprodutibilidade.

## 1. Gerador do conjunto sintético

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# define a função que gera o dataset sintético
def gerar_dataset_sintetico(n=200, seed=42, data_referencia="2025-01-01"):
    # fixa a semente para reprodutibilidade
    np.random.seed(seed)
    # lista completa de UFs brasileiras
    ufs = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB",
           "PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]
    # usa uma data fixa como referência para que as idades sejam reproduzíveis
    hoje = pd.Timestamp(data_referencia).to_pydatetime()
    # gera datas de nascimento aleatórias entre 18 e 40 anos
    datas_nascimento = [hoje - timedelta(days=int(365 * np.random.uniform(18, 40))) for _ in range(n)]
    # cria o DataFrame principal
    df = pd.DataFrame({
        # cria um identificador único incremental
        "id": range(1, n + 1),
        # gera sexo aleatório
        "sexo": np.random.choice(["M", "F"], size=n),
        # gera estados aleatórios
        "uf": np.random.choice(ufs, size=n),
        # atribui datas de nascimento
        "data_nascimento": datas_nascimento,
        # gera notas de matemática com distribuição normal truncada
        "nota_matematica": np.random.normal(6.5, 1.5, n).clip(0, 10),
        # gera notas de física com distribuição normal truncada
        "nota_fisica": np.random.normal(7.5, 2.7, n).clip(0, 10),
        # gera notas de química com distribuição normal truncada
        "nota_quimica": np.random.normal(6.5, 1.6, n).clip(0, 10),
        # gera notas de inglês com distribuição normal truncada
        "nota_ingles": np.random.normal(7.0, 1.2, n).clip(0, 10),
        # gera notas de português com distribuição normal truncada
        "nota_portugues": np.random.normal(6.8, 1.3, n).clip(0, 10),
        # gera quantidade de filhos como inteiro (Poisson)
        "quantidade_filhos": np.random.poisson(lam=1.2, size=n).astype(int)
    })
    # calcula a idade aproximada a partir da data de nascimento
    df["idade"] = df["data_nascimento"].apply(lambda x: (hoje - x).days // 365)
    # força zero filhos para menores de 18 anos
    df.loc[df["idade"] < 18, "quantidade_filhos"] = 0
    # retorna o DataFrame final
    return df

# Gera o dataset sintético

In [ ]:
# gera o dataset sintético
df = gerar_dataset_sintetico(n=20000)
#
df.info()

# Comandos Python

✔️ import pandas as pd  
✔️ pd.read_csv()  
✔️ df.head()  
✔️ df.tail()  
✔️ df.info()  
✔️ df.describe()  
✔️ df['coluna']  
✔️ df[df['coluna'] > x]  
✔️ df.groupby()  
✔️ df.sort_values()  
✔️ df.isnull().sum()  
✔️ df.fillna()  
✔️ df.dropna()

## Head

In [ ]:
# exibe as primeiras linhas do dataset
df.head(3)

## Tail

In [ ]:
# exibe as últimas linhas do dataset
df.tail(2)

## Info

In [ ]:
# mostra tipos, memória e valores nulos
df.info()

## Describe

In [ ]:
# apresenta estatísticas descritivas das colunas numéricas
df.describe()

## df[nome_coluna]

In [ ]:
df.head(2)

In [ ]:
# seleciona uma única coluna
df["uf"]

## Filtro lógico com nome_coluna

In [ ]:
# filtra registros com idade maior que 25
df[df["idade"] > 25]

## Group by com média

In [ ]:
# agrupa por estado e calcula média de matemática
df.groupby("uf")["nota_matematica"].mean()

## Group by com mais de uma coluna e média

In [ ]:
# agrupa por sexo e calcula médias de física e química
df.groupby("sexo")[["nota_fisica", "nota_quimica"]].mean().reset_index()

## Group by (bônus-usando agg)

In [ ]:
tb = df.groupby(['uf','sexo'], dropna = False
          ).agg(media_matematica = ('nota_matematica','mean'),
                desvpad_matematica = ('nota_matematica','std'),
                min_matematica = ('nota_matematica','min')
               ).reset_index()

## Sort_values

In [ ]:
# ordena os alunos pela nota de matemática
df.sort_values("nota_matematica", ascending=False)

## Isnull() e sum()

In [ ]:
# contabiliza valores nulos por coluna
df.isnull().sum()

## Fillna

In [ ]:
# substitui valores nulos
df.fillna(0)

## Dropna()

In [ ]:
# remove linhas com qualquer valor nulo
df.dropna()

## Criando arquivo parquet

In [ ]:
df.to_parquet('df_fast_track.parquet')

## Criando arquivo csv

In [ ]:
df.to_csv('df_fast_track.csv', sep = ';', index = False)

## Lendo arquivo csv

In [ ]:
df.head(2)

In [ ]:
df_from_csv = pd.read_csv('df_fast_track.csv', sep = ';')
df_from_csv.head()

# Comandos SQL (com duckdb)
✔️ SELECT  
✔️ FROM  
✔️ WHERE  
✔️ ORDER BY  
✔️ SUM()  
✔️ COUNT()  
✔️ AVG()  
✔️ GROUP BY  
✔️ CREATE  
✔️ INSERT  
✔️ DELETE

## Importando o duckdb

In [ ]:
# importa a biblioteca duckdb
import duckdb
# cria uma conexão em memória
con = duckdb.connect()
# registra o DataFrame como tabela SQL
con.register("alunos", df)

## Select, from

In [ ]:
# executa um SELECT com filtro por idade
con.execute("SELECT * FROM alunos limit 5").df()

## From, group by, order by

In [ ]:
df

In [ ]:
con.execute("""
select 
uf
,count(id) as qtd_alunos
,avg(nota_fisica) as media_fisica
,avg(nota_matematica) as media_matematica
from alunos
group by 1
order by 4 desc
""").df()

In [ ]:
# executa agregação com GROUP BY e ORDER BY
con.execute("""
SELECT
    uf,
    COUNT(*) AS total_alunos,
    AVG(nota_matematica) AS media_matematica
FROM alunos
GROUP BY uf
ORDER BY media_matematica DESC
""").df()

## Select, from, where

In [ ]:
con.execute("SELECT * FROM alunos WHERE nota_fisica < 5 ").df()

In [ ]:
con.execute("SELECT * FROM alunos WHERE uf = 'AC' ").df()

## Create table

In [ ]:
# cria uma tabela física a partir do DataFrame
con.execute("CREATE TABLE alunos_sql AS SELECT * FROM alunos")

In [ ]:
con.execute("DESCRIBE alunos_sql").fetchdf()

## Insert

In [ ]:
# insere um novo registro manualmente
con.execute("INSERT INTO alunos_sql VALUES (20001,'F','SP',DATE '2000-01-01',8.0,7.5,7.0,9.0,8.5,1,24)")

In [ ]:
con.execute("SELECT * FROM alunos_sql where id = 20001").df()

## Delete

In [ ]:
# remove registros com nota de física nula
con.execute("DELETE FROM alunos_sql WHERE id = 20001")

In [ ]:
con.execute("SELECT * FROM alunos_sql WHERE id = 20001").df()

# Funções do PowerBI
✔️ `SUM(coluna)`
Soma todos os valores numéricos da coluna no contexto atual de filtro.  
✔️ `COUNT(coluna)`
Conta quantos valores não nulos existem na coluna, respeitando os filtros.  
✔️ `DIVIDE(numerador, denominador)`
Realiza divisão segura, evitando erro quando o denominador é zero.  
✔️ `CALCULATE()`
Recalcula uma métrica alterando o contexto de filtro. É a função central do DAX.  
✔️ `FILTER()`
Cria um filtro linha a linha com base em uma condição lógica.  
✔️ `ALL()`
Remove filtros de uma tabela ou coluna, permitindo cálculos globais.  
✔️ `VALUES()`
Retorna os valores distintos de uma coluna dentro do contexto atual.  
✔️ `COUNTROWS()`
Conta o número de linhas de uma tabela (física ou virtual).  
✔️ `IF()`
Avalia uma condição sobre métricas agregadas e retorna um valor conforme o resultado.  
✔️ `SWITCH()`
Avalia múltiplas condições sobre uma métrica agregada, substituindo vários IF encadeados.  
✔️ `SUMX()`
Avalia uma expressão linha a linha e soma os resultados obtidos.  
✔️ `RELATED()`
Busca um valor em uma tabela relacionada. Usado apenas em coluna calculada.